# 📝 벡터 검색 과제 LV2(응용): 검색에 그래프를 얹기

> 의미 검색에 **그래프 구조**(PageRank·커뮤니티)를 얹어 순위를 다듬고 범위를 좁힙니다. 교안에서는 약물 대사를, LV1 에서는 고혈압을 물었습니다. 여기서는 **류마티스 관절염** 쪽으로 묻습니다. 문제는 네 갈래로 묶여 있습니다.
>
> - **1. 검색으로 후보 뽑기**: `SEARCH` 절로 직접 · `VectorRetriever` 로 같은 일
> - **2. 그래프 점수 만들기**: `MENTIONS` 로 개체 보기 · 사전이 안 이은 것 손으로 잇기 · 투영과 `gds.pageRank.write` · 개체 점수를 문서 점수로(평균과 바닥값)
> - **3. 그래프로 검색 결과 손보기**: 눈금 맞추기(`minmax`)와 가중합 리랭킹 · `gds.leiden.write` 로 범위 좁히기 · 둘의 차이
> - **4. 찾은 근거로 답 만들기**: 발췌와 개체를 묶어 근거로 · 답을 받아 인용 대조 · 서술형

## 풀이 방법
1. 맨 위 **준비 셀**을 먼저 실행하세요(그래프·논문·임베딩·인덱스·`MENTIONS` 가 모두 준비됩니다).
2. 문제는 **순서대로** 푸세요. 앞 문제의 결과를 뒤 문제가 씁니다.
3. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- **분량**: 코딩 10문 + 서술형 1문(4항목). 준비 셀부터 끝까지 110분 안팎 걸립니다. 답변 생성은 4-2번에서 한 번 부릅니다.
- **이 과제는 실행 중인 Neo4j 가 필요합니다.** 반드시 **실습 전용** DB 에 연결하세요.
- 자가채점은 여러분이 담은 값을 **그 자리에서 다시 조회한 결과와 대조**합니다. 실제로 돌려야 통과합니다.

화이팅!

> **데이터 출처**: 이 단원의 데이터는 **공개된 원본을 값 그대로** 쓴 것입니다.
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 69편 (`pmc_docs.jsonl`) | PubMed Central Open Access Subset. 각 행의 `pmcid` 가 원문 주소다 | **CC BY** |
> | 의료 지식그래프 (`hetionet_*.csv`) | Hetionet v1.0 (https://het.io) 에서 CC0 출처만 골라낸 부분 | **CC0** |
> | 이름 사전 (`name2id.json.gz`) | 위 Hetionet 이름 + RxNav(미국 국립의학도서관) 약물 동의어 | CC0 · NLM |
>
> 지식그래프는 **2016년에 정리된 자료**이고, 논문은 최근 것입니다. 그래서 이 둘을 이어 붙이면 그래프가 모르는 사실이 논문 쪽에 있습니다. 이 단원은 그 상태 그대로 검색합니다.
>
> 그리고 **논문이 보고했다**와 **효능이 입증됐다**는 다릅니다. 검색으로 찾은 문장을 답으로 옮길 때 이 구분을 놓치면, 근거가 있는 것처럼 보이는 틀린 답이 나옵니다.

아래 준비 셀을 먼저 실행하세요(그래프 적재에 몇 초 걸립니다).

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 임베딩 준비: embed_texts(문장 리스트) 가 768차원 OpenAI 임베딩을 돌려줍니다(실행만 하세요).
# embed_texts(문서 리스트) 는 계산한 벡터를 data/emb_cache.pkl 에 저장해 두고 다시 씁니다(69편을 매번 다시 부르지 않으려고요).
# embed_query(질문 한 문장) 는 저장하지 않습니다. 질문은 매번 새로 만드는 것이 표준입니다.
import pickle
from pathlib import Path

from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 768                    # 3072차원으로 나오는 모델을 768차원으로 잘라 받는다(아래 dimensions)
_EMB_FILE = Path("data/emb_cache.pkl") if Path("data").exists() else Path("../data/emb_cache.pkl")
_EMB_CACHE = pickle.loads(_EMB_FILE.read_bytes()) if _EMB_FILE.exists() else {}   # {문서: 벡터}
embedder = OpenAIEmbeddings(model=EMBED_MODEL, dimensions=EMBED_DIM)


def embed_texts(texts):
    """문서 리스트 -> 768차원 임베딩 리스트. 저장된 것은 그대로 쓰고, 없는 것만 임베딩해 저장한다."""
    new = [t for t in texts if t not in _EMB_CACHE]                  # 저장돼 있지 않은 문서만 고른다
    if new:
        _EMB_CACHE.update(zip(new, embedder.embed_documents(new)))   # 실제 호출은 이 줄뿐
        _EMB_FILE.write_bytes(pickle.dumps(_EMB_CACHE))              # 통째로 다시 저장
    return [_EMB_CACHE[t] for t in texts]


def embed_query(text):
    """질문 한 문장 -> 768차원 임베딩. 질문은 저장하지 않는다(매번 새로 만든다)."""
    return embedder.embed_query(text)


print("임베딩 모델:", EMBED_MODEL, f"({EMBED_DIM}차원) / 저장된 문서:", len(_EMB_CACHE), "건")

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 이 셀은 실행만 하세요.
# 몇 번이든 처음부터 다시 돌릴 수 있게 전부 내립니다. 반드시 "실습 전용" DB 여야 합니다.

# 1) GDS 투영: 노드를 지우기 전에 먼저 내린다. 투영은 원본 노드 id 를 기억하고 있어, 원본을 먼저 지우면 갈 곳을 잃는다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName", name=_g["graphName"])

# 2) 노드와 관계
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

# 3) 벡터·전문 인덱스: 노드를 지워도 인덱스는 남는다. 차원이 다른 옛 인덱스가 남아 있으면 뒤에서 걸린다
for _ix in run_cypher("SHOW INDEXES YIELD name, type WHERE type IN ['VECTOR','FULLTEXT'] RETURN name"):
    run_cypher(f"DROP INDEX {_ix['name']} IF EXISTS")   # IF EXISTS: 이미 없어도 에러 없이 넘어간다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 의료 지식 그래프 적재: 이 셀은 실행만 하세요(2초쯤 걸립니다).
# 32일차에서 적재한 그 그래프입니다(Hetionet v1.0 중 재배포 가능한 CC0 부분, 2016년 자료).
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")   # 정답 폴더에서도 돌게
NODE_LABELS = ['Compound', 'Disease', 'Gene', 'Symptom', 'PharmacologicClass']
REL_ENDS = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_CC": ("Compound", "Compound"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

for _label in NODE_LABELS:
    run_cypher(f"CREATE INDEX {_label.lower()}_id IF NOT EXISTS FOR (n:{_label}) ON (n.id)")

_nodes = {_label: [] for _label in NODE_LABELS}
for _row in pd.read_csv(DATA_DIR / "hetionet_nodes.csv").to_dict("records"):
    _nodes[_row["label"]].append({"id": _row["id"], "name": _row["name"]})
for _label, _rows in _nodes.items():
    run_cypher(f"UNWIND $rows AS row CREATE (n:{_label}) SET n.id = row.id, n.name = row.name",
               rows=_rows)

_edges = {_rel: [] for _rel in REL_ENDS}
for _row in pd.read_csv(DATA_DIR / "hetionet_edges.csv").to_dict("records"):
    _edges[_row["rel"]].append({"s": _row["source"], "t": _row["target"]})
for _rel, _rows in _edges.items():
    _src, _dst = REL_ENDS[_rel]
    for _start in range(0, len(_rows), 20000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{_src} {{id: row.s}}), (b:{_dst} {{id: row.t}}) "
                   f"CREATE (a)-[:{_rel}]->(b)", rows=_rows[_start:_start + 20000])

print("노드:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"],
      "/ 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])


In [ ]:
# [제공 코드] 검색 준비 완료 상태 만들기: 적재 -> 임베딩 -> 벡터/전문 인덱스 -> MENTIONS (실행만 하세요).
import gzip
import json
import re
from pathlib import Path

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
papers = [json.loads(_line) for _line in
          (DATA_DIR / "pmc_docs.jsonl").read_text(encoding="utf-8").splitlines() if _line.strip()]
run_cypher("CREATE INDEX document_pmcid IF NOT EXISTS FOR (n:Document) ON (n.pmcid)")
run_cypher("UNWIND $rows AS row CREATE (n:Document) SET n += row", rows=papers)

_vectors = embed_texts([_p["title"] + " " + _p["text"] for _p in papers])
run_cypher("""UNWIND $rows AS row
              MATCH (n:Document {pmcid: row.pmcid})
              CALL db.create.setNodeVectorProperty(n, 'emb', row.vec)""",
           rows=[{"pmcid": _p["pmcid"], "vec": _vec} for _p, _vec in zip(papers, _vectors)])
run_cypher("""CREATE VECTOR INDEX doc_vec IF NOT EXISTS FOR (n:Document) ON n.emb
             OPTIONS {indexConfig: {`vector.dimensions`: 768,
                                    `vector.similarity_function`: 'cosine'}}""")
run_cypher("CREATE FULLTEXT INDEX doc_ft IF NOT EXISTS FOR (n:Document) ON EACH [n.title, n.text]")
run_cypher("CALL db.awaitIndexes()")

with gzip.open(DATA_DIR / "name2id.json.gz", "rt", encoding="utf-8") as _f:
    NAME2ID = json.load(_f)

TOKEN = re.compile(r"[A-Za-z][A-Za-z0-9'-]*")   # 한 단어의 모양: 영문으로 시작하고 숫자·따옴표·하이픈까지 한 단어로 본다


def find_entities(text, name2id):
    """문서에서 사전에 있는 이름을 찾아 {id: (레이블, 표준 이름)} 으로 돌려줍니다."""
    entries, genes = name2id["entries"], name2id["genes"]
    ambiguous, brand_stopwords = name2id["ambiguous"], name2id["brand_stopwords"]
    found = {}   # 키가 노드 id 라 같은 개체를 두 번 잡아도 한 번만 남는다
    for word in TOKEN.findall(text):
        # 유전자 기호는 대소문자를 그대로 맞춥니다(소문자로 누르면 평범한 단어가 유전자가 됩니다)
        if word in genes:
            found[genes[word]] = ("Gene", word)
            continue
        name = word.lower()   # 나머지 이름은 소문자로 맞춰 사전을 찾습니다
        # ambiguous: 종류가 다른 노드 둘에 걸리는 이름 9개. 잘못 합칠 바에는 안 잇습니다
        if name not in entries or name in ambiguous:
            continue
        # 흔한 영어 단어와 겹치는 상품명은 원래 대소문자로 쓰인 자리만 인정합니다
        if name in brand_stopwords and word.islower():
            continue
        entry = entries[name]
        found[entry["id"]] = (entry["label"], entry["canonical"])
    return found


_links = {}
for _p in papers:
    for _eid, (_label, _canon) in find_entities(_p["title"] + " " + _p["text"], NAME2ID).items():
        _links.setdefault(_label, []).append({"pmcid": _p["pmcid"], "id": _eid})
for _label, _rows in _links.items():
    run_cypher(f"UNWIND $rows AS row MATCH (d:Document {{pmcid: row.pmcid}}), "
               f"(e:{_label} {{id: row.id}}) MERGE (d)-[:MENTIONS]->(e)", rows=_rows)

print("준비 완료: 논문", len(papers), "편 · MENTIONS",
      run_cypher("MATCH (:Document)-[r:MENTIONS]->() RETURN count(r) AS c")[0]["c"], "건")


In [ ]:
# [제공 코드] 검색기용 임베더 어댑터: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
from neo4j_graphrag.embeddings.base import Embedder


class QueryEmbedder(Embedder):
    """검색기가 요구하는 embed_query 자리에 준비 셀의 embedder 를 끼워 넣는 어댑터."""

    def embed_query(self, text):
        """질문 한 문장을 문서와 같은 768차원으로."""
        return embedder.embed_query(text)   # 준비 셀에서 dimensions=768 로 만들어 둔 그 embedder


query_embedder = QueryEmbedder()

print("임베더 어댑터 준비 완료")

In [ ]:
# [제공 코드] 답변 프롬프트 틀: 이 셀은 실행만 하세요.
# 교안_02 4-2 의 두 규칙(지어내지 말 것·문장마다 pmcid 대괄호)을 한 틀에 적은 것입니다.
# 이 규칙이 4-2 채점의 전제라 틀을 그대로 드립니다.
# 채울 빈칸은 context 와 question 입니다.
from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_template(
    "다음 논문 발췌를 근거로 질문에 한국어 세 문장 안으로 답하세요.\n"
    "규칙 두 가지를 지키세요.\n"
    "1) 발췌에 없는 내용은 지어내지 마세요.\n"
    "2) 각 문장 끝에 근거가 된 논문의 pmcid 를 대괄호로 다세요.\n\n"
    "[논문]\n{context}\n\n[질문] {question}\n[답변]")

print("프롬프트 틀 준비 완료. 채울 빈칸:", sorted(answer_prompt.input_variables))


## 데이터 살펴보기

문제를 풀기 전에 무엇이 준비됐는지 한 번 훑고 갑니다.

In [ ]:
# [제공 코드] 준비된 것 훑어보기: 이 셀은 실행만 하세요.
# 논문·개체·다리(MENTIONS)가 각각 몇 개인지, 인덱스는 무엇이 서 있는지 확인한다
for row in run_cypher('MATCH (n) RETURN labels(n)[0] AS label, count(*) AS cnt ORDER BY cnt DESC'):
    print(f"  {row['label']:20} {row['cnt']:>6,}")
print('MENTIONS:', run_cypher('MATCH (:Document)-[r:MENTIONS]->() '
                              'RETURN count(r) AS c')[0]['c'], '건')
print('인덱스:', [row['name'] for row in
                run_cypher('SHOW INDEXES YIELD name, type WHERE type IN ["VECTOR", "FULLTEXT"] '
                           'RETURN name ORDER BY name')])

---
# 1. 검색으로 후보 뽑기

질문 하나를 두 방법으로 검색해 후보 다섯 편을 뽑는 연습입니다(교안_02 1절). 직접 쓴 `SEARCH` 절과 라이브러리 검색기가 같은 답을 내는지 확인합니다.

## 1-1. 의미로 논문 찾기
**배경**: 류마티스 관절염 담당자가 **"류마티스 관절염 치료에 쓰는 약"** 을 찾고 있습니다. 뒤 문제에서 이 결과를 계속 쓰므로, 점수와 그래프 점수까지 함께 받아 둡니다.

**요구사항**:
- 질문 **`'류마티스 관절염 치료에 쓰는 약'`** 로 `doc_vec` 에서 **상위 5편**을 찾으세요.
- 각 행에 `pmcid`·`score`·`graph_score`(노드의 `graph_score` 속성) 를 담아 **`hits`** 에 리스트로 두세요. `score` 는 `SEARCH` 가 돌려준 값을 **반올림하지 말고 그대로** 담으세요.
- `pmcid` 만 뽑아 **`top_ids`** 에 담고 출력하세요.

**확인 기준**: `hits` 는 유사도 내림차순이고(`score` 는 소수 여섯째 자리까지 대조합니다), `top_ids` 는 길이 5의 문자열 리스트입니다. 1위는 메토트렉세이트가 림프구에 미치는 영향이 아니라 홍경천(Rhodiola rosea)의 류마티스 관절염 치료 효과를 다룬 논문입니다. `graph_score` 는 아직 만들지 않았으므로 지금은 전부 `None` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문을 임베딩하고, MATCH 로 잡은 Document 를 SEARCH 절에 넘겨 상위 5편을 받는다.

세부구현:
1. 질문 문장을 embed_query 에 넘겨 벡터를 얻는다(질문은 저장하지 않는다).
2. MATCH 로 잡은 노드를 SEARCH 절에 넘긴다. 괄호 안에 인덱스 이름·질문 벡터·LIMIT,
   닫는 괄호 뒤에 점수 별칭을 쓴다(교안_01 3-2).
3. RETURN 에 pmcid·score·graph_score 세 값을 별칭 그대로 담고 score 내림차순으로 정렬한다.
4. 결과 리스트를 hits 에 두고, pmcid 만 뽑아 top_ids 를 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(hits[0]) >= {'pmcid', 'score', 'graph_score'}, 'hits 의 각 행에 세 키를 모두 담으세요'
assert len(top_ids) == 5, f'top_ids 는 5편이어야 합니다(현재 {len(top_ids)}편). LIMIT 을 확인하세요'
# 목록만 보면 손으로 적어도 통과한다. 그 자리에서 다시 검색해 순서와 점수까지 대조한다
truth = run_cypher('''MATCH (n:Document)
                        SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 5) SCORE AS score
                      RETURN n.pmcid AS pmcid, score ORDER BY score DESC''',
                   q=embed_query('류마티스 관절염 치료에 쓰는 약'))
assert [hit['pmcid'] for hit in hits] == [row['pmcid'] for row in truth], \
    'hits 를 검색 결과 그대로(같은 순서로) 담으세요'
assert top_ids == [hit['pmcid'] for hit in hits], 'top_ids 는 hits 에서 pmcid 만 뽑은 것입니다'
# 5위와 6위의 점수 차가 0.002 밖에 안 돼, 다시 적재하면 5위가 갈릴 수 있다. 리터럴로는 1위만 본다
assert top_ids[0] == 'PMC13493261', (f'1위가 {top_ids[0]} 로 나왔습니다. 이 코퍼스의 답은 PMC13493261 입니다. 질문 문장을 확인하세요')
assert isinstance(hits[0]['score'], float), 'score 자리에 SEARCH 가 돌려준 유사도를 담으세요'
assert abs(hits[0]['score'] - truth[0]['score']) < 1e-6, \
    'score 를 직접 적지 말고 SCORE AS 로 받은 값을 그대로 담으세요'
print('✅ 통과!')

## 1-2. 같은 검색을 리트리버로 하기
**배경**: 라이브러리 검색기(`VectorRetriever`)는 질문 문장만 주면 검색까지 해 줍니다. 1-1번과 같은 답이 나오는지 확인합니다.

**요구사항**:
- `doc_vec` 인덱스로 `VectorRetriever` 를 **`retriever`** 라는 이름으로 만드세요(`return_properties=['pmcid']`, 임베더는 `query_embedder`).
- 1-1번과 **같은 질문**으로 `top_k=5` 검색하세요.
- 각 항목의 `content` 에서 `pmcid` 만 골라 **`retriever_ids`** 에 리스트로 담고 출력하세요.

**확인 기준**: `retriever_ids` 가 1-1번의 `top_ids` 와 **완전히 같습니다.** `content` 는 `{'pmcid': 'PMC...'}` 모양의 **문자열**이라 대괄호로 꺼낼 수 없습니다. 문자열에서 골라내야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색기를 만들어 search 를 부르고, 각 항목의 content 문자열에서 PMC 로 시작하는 토막을 골라낸다.

세부구현:
1. neo4j_graphrag.retrievers 의 VectorRetriever 에 드라이버·인덱스 이름·임베더·돌려받을 속성을
   넘겨 검색기를 만든다(교안_02 1-1).
3. search 에 query_text 와 top_k 를 넘긴다. 결과의 items 를 돈다.
4. content 는 문자열이다. 정규식으로 PMC 와 이어지는 숫자를 찾거나, 따옴표로 잘라 값을 꺼낸다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import re as _re

assert retriever_ids == top_ids, (f'리트리버 결과({retriever_ids})가 1-1번({top_ids})과 다릅니다. '
                                  '같은 질문·같은 인덱스·같은 top_k 인지 확인하세요')
# 목록만 보면 top_ids 를 그대로 베껴도 통과한다. 검색기를 그 자리에서 다시 두드려 본다
_probe = retriever.search(query_text='류마티스 관절염 치료에 쓰는 약', top_k=3)
_found = [_re.search(r'PMC\d+', _item.content) for _item in _probe.items]
assert all(_found), ('검색기가 돌려준 content 에 pmcid 가 없습니다. '
                    'result_formatter 를 달았다면 content 가 발췌 문자열이 됩니다. '
                    '이 문제는 return_properties=[\'pmcid\'] 로 푸세요')
assert [_m.group(0) for _m in _found] == top_ids[:3], \
    'VectorRetriever 를 실제로 만들어 search 로 받으세요'
print('✅ 통과!')

---
# 2. 그래프 점수 만들기

찾은 논문에서 그래프로 건너가는 **다리**를 확인하고, 사전이 안 이은 자리를 손으로 메운 뒤, 개체에 중심성 점수를 새겨 그것을 문서 점수로 옮기는 연습입니다(교안_01 5절 · 교안_02 2-1·2-2).

## 2-1. 찾은 논문에서 그래프로 넘어가기
**배경**: 검색이 데려다준 논문이 **그래프의 어떤 개체를 가리키는지** 보고, 거기서 **한 걸음 더** 건너가 봅니다(교안_01 5-3).

**요구사항**:
- 1-1번 결과의 **1위 논문**(`top_ids[0]`)이 `MENTIONS` 로 가리키는 개체를 조회하세요.
- **레이블별 개수**를 `{'Compound': n, ...}` 모양의 딕셔너리로 **`entity_counts`** 에 담으세요.
- `entity_counts` 와 그 논문이 언급한 개체 **이름 목록**을 출력하세요.
- 그 논문이 언급한 **약물**에서 `BINDS` 로 한 걸음 더 건너가, 붙어 있는 **유전자 이름**을 중복 없이 **이름 순**으로 **`hop_genes`** 에 리스트로 담고 개수를 출력하세요.

**확인 기준**: `entity_counts` 는 딕셔너리이고 값의 합이 그 논문의 `MENTIONS` 관계 수와 같습니다. 레이블은 **두 종류**(`Compound`·`Gene`)이고 개체는 셋입니다. 류마티스 관절염 같은 질병 이름은 붙지 않습니다. `hop_genes` 는 문자열 리스트이고, 이 논문이 직접 언급한 유전자(`CRP`)는 그 안에 **없습니다.**

<details><summary>힌트</summary>

```text
접근방법:
- Document 에서 MENTIONS 로 이어진 노드를 잡아, 레이블별로 세고 이름도 모은다.

세부구현:
1. MATCH 로 그 pmcid 의 Document 와 MENTIONS 로 이어진 노드를 함께 잡는다.
2. 레이블은 labels(e)[0] 으로 꺼낸다. RETURN 에서 레이블로 묶고 count 로 센다.
3. 결과 행을 파이썬 딕셔너리로 옮긴다.
4. 이름 목록은 같은 패턴을 다시 조회하거나 collect 로 함께 받는다.
5. 두 걸음은 MENTIONS 뒤에 BINDS 를 이어 붙인다. DISTINCT 로 중복을 없애고 이름 순으로 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
total = run_cypher('MATCH (d:Document {pmcid:$pmcid})-[r:MENTIONS]->() RETURN count(r) AS c',
                   pmcid=top_ids[0])[0]['c']
assert isinstance(entity_counts, dict), 'entity_counts 는 딕셔너리여야 합니다'
assert sum(entity_counts.values()) == total, (f'합이 {sum(entity_counts.values())}인데 실제 MENTIONS 는 '
                                              f'{total}건입니다. 레이블별로 빠짐없이 세세요')
# 딕셔너리를 손으로 적어도 통과하지 않도록, 그 자리에서 다시 세어 대조한다
truth = {row['label']: row['cnt'] for row in run_cypher(
    '''MATCH (d:Document {pmcid:$pmcid})-[:MENTIONS]->(e)
       RETURN labels(e)[0] AS label, count(*) AS cnt''', pmcid=top_ids[0])}
assert entity_counts == truth, f'레이블별 개수가 다릅니다: {entity_counts} (실제 {truth})'
# 두 걸음도 그 자리에서 다시 건너가 대조한다
_hop = [row['name'] for row in run_cypher(
    '''MATCH (d:Document {pmcid:$pmcid})-[:MENTIONS]->(c:Compound)-[:BINDS]->(g:Gene)
       RETURN DISTINCT g.name AS name ORDER BY name''', pmcid=top_ids[0])]
assert hop_genes == _hop, (f'hop_genes 가 {len(hop_genes)}개인데 실제로 건너가면 {len(_hop)}개입니다. '
                           'MENTIONS 뒤에 BINDS 를 이어 붙이고 DISTINCT·이름 순으로 담으세요')
print('✅ 통과!')

## 2-2. 사전이 안 이은 자리를 손으로 잇기
**배경**: 교안_01 5-1 에서 본 **애매한 이름 아홉 개**는 사전이 아예 잇지 않습니다. 아래 세 쌍은 사람이 논문을 읽고 어느 쪽인지 정해 둔 것입니다.

**요구사항**:
아래 목록을 답안 셀에 복사해 두세요.

```python
CURATED = [
    {'pmcid': 'PMC13493084', 'label': 'Disease',  'id': 'Disease::DOID:9970'},
    {'pmcid': 'PMC13496164', 'label': 'Disease',  'id': 'Disease::DOID:9970'},
    {'pmcid': 'PMC13490353', 'label': 'Compound', 'id': 'Compound::DB00396'},
]
```

- 각 쌍을 `MENTIONS` 로 이으세요. **이름이 아니라 `id` 로** 찾고, **`MERGE`** 로 이으세요(두 번 실행해도 관계가 하나여야 합니다).
- **레이블별로 나눠** 보내세요. `MATCH` 에 레이블을 찍어 줘야 `id` 인덱스를 탑니다.
- 이은 뒤 전체 `MENTIONS` 건수를 세어 **`mention_total`** 에 담고 출력하세요.

**확인 기준**: 준비 셀이 만든 `MENTIONS` 는 **588건**이고, 세 쌍을 이으면 **591건**이 됩니다. 답안 셀을 두 번 실행해도 591건 그대로여야 합니다(`MERGE` 를 썼다면 그렇습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 레이블이 같은 것끼리 모아, 레이블마다 한 번씩 UNWIND + MERGE 질의를 보낸다.

세부구현:
1. CURATED 를 label 값으로 묶어 딕셔너리에 담는다(교안_01 5-2 의 links 와 같은 모양).
2. 레이블마다 f-string 으로 MATCH (e:레이블 {id: row.id}) 를 만든다. 레이블은 파라미터로 못 넘긴다.
3. MATCH 로 Document 와 개체를 함께 잡고 MERGE 로 관계를 만든다.
4. MATCH (:Document)-[r:MENTIONS]->() 로 전체 건수를 센다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
for _pair in [
    ('PMC13493084', 'Disease', 'Disease::DOID:9970'),
    ('PMC13496164', 'Disease', 'Disease::DOID:9970'),
    ('PMC13490353', 'Compound', 'Compound::DB00396'),
]:
    _p, _label, _id = _pair
    _found = run_cypher(f'MATCH (d:Document {{pmcid:$p}})-[r:MENTIONS]->(e:{_label} {{id:$i}}) '
                        'RETURN count(r) AS c', p=_p, i=_id)[0]['c']
    assert _found == 1, f'{_p} -> {_id} 가 {_found}건입니다. MERGE 로 한 번만 이으세요'
# 세 쌍만 늘어야 한다. 전체를 통째로 이어 버리면 여기서 걸린다
_real = run_cypher('MATCH (:Document)-[r:MENTIONS]->() RETURN count(r) AS c')[0]['c']
assert _real == 591, \
    f'전체 MENTIONS 가 {_real}건입니다. 준비 셀의 588건 + 세 쌍이어야 합니다'
assert mention_total == _real, 'mention_total 을 DB 에서 실제로 센 값으로 담으세요'
print('✅ 통과!')

## 2-3. 투영을 만들고 PageRank 새기기
**배경**: 검색 결과를 다듬으려면 **개체가 그래프에서 얼마나 중심에 있는지**가 필요합니다. 33일차에서 쓰던 약물 중심 투영을 다시 만들고 PageRank 를 계산합니다.

**요구사항**:
- 이름 **`drugGraph`** 로 투영을 만드세요. 노드 레이블은 **`Compound`, `Disease`, `PharmacologicClass`**, 관계는 **`TREATS`, `PALLIATES`, `INCLUDES`, `RESEMBLES_DD`, `RESEMBLES_CC`** 이고 전부 **무방향**입니다.
- `gds.pageRank.write` 로 개체 노드의 **`pagerank`** 속성에 점수를 새기세요.
- 점수가 새겨진 노드 수를 **`pr_written`**, PageRank 1위 노드의 이름을 **`pr_top`** 에 담으세요.

**확인 기준**: `pr_written` 은 투영의 노드 수와 같고, `pr_top` 은 그 투영에서 PageRank 가 가장 높은 노드의 `name` 입니다.

> **이 셀을 다시 돌리려면** 먼저 `CALL gds.graph.drop('drugGraph', false) YIELD graphName RETURN graphName` 으로 같은 이름 투영을 내리세요(투영 이름은 겹칠 수 없습니다). 맨 위 준비 셀로 돌아가면 2-2번에서 이은 관계까지 지워집니다.

<details><summary>힌트</summary>

```text
접근방법:
- gds.graph.project 로 투영을 만들고, gds.pageRank.write 로 점수를 노드 속성에 쓴다. 그다음 그 속성을 조회한다.

세부구현:
1. 투영 프로시저에 이름·레이블 리스트·관계 설정을 넘긴다. 무방향은 관계마다 orientation 을 지정한다.
2. 프로시저 호출은 YIELD 로 쓸 컬럼만 골라 받는다(교안_02 2-1 과 같은 모양).
3. pageRank.write 는 두 번째 인자 맵에 writeProperty 를 준다.
4. pagerank 가 NULL 이 아닌 노드를 세고, 내림차순 첫 행의 이름을 꺼낸다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
real_written = run_cypher('MATCH (n) WHERE n.pagerank IS NOT NULL RETURN count(n) AS c')[0]['c']
assert real_written == 2012, (f'pagerank 가 새겨진 노드가 {real_written}개입니다. '
                                     '투영에 담은 레이블과 관계를 확인하세요(2012개여야 합니다)')
assert pr_written == real_written, 'pr_written 을 DB 에서 실제로 센 값으로 담으세요'
assert pr_top == 'hypertension', f'1위가 다릅니다: {pr_top}. 무방향으로 투영했는지 확인하세요'
print('✅ 통과!')

## 2-4. 개체 점수를 문서 점수로 옮기기
**배경**: PageRank 는 **개체**에 붙었는데 검색 결과는 **문서** 단위라 점수를 옮겨야 합니다.

**요구사항**:
- 각 `Document` 에 대해, 그 문서가 `MENTIONS` 로 가리키면서 `pagerank` 가 있는 개체들의 **평균 PageRank** 를 `graph_score` 속성에 쓰세요.
- 점수를 받지 못한 문서(투영에 든 개체를 하나도 안 쓴 문서)에는 **`0.15`**(PageRank 의 바닥값)를 쓰세요.
- `graph_score` 가 `0.15` 보다 큰 문서 수를 **`scored`** 에 담으세요.

**확인 기준**: `scored` 는 정수이고 **59** 입니다(2-2번의 세 쌍 덕에 교안_02 의 58편보다 한 편 많습니다).

<details><summary>힌트</summary>

```text
접근방법:
- MENTIONS 로 이어진 개체 중 pagerank 가 있는 것만 모아 평균을 내고 문서에 SET 한다. 그다음 빈 문서를 채운다.

세부구현:
1. MATCH 로 Document 와 MENTIONS 로 이어진 개체를 잡고, WHERE 로 pagerank 가 NULL 이 아닌 것만 남긴다.
2. WITH 로 문서별 평균을 낸다. 평균 함수 이름은 avg 다.
3. SET 으로 그 값을 문서 속성에 쓴다.
4. 두 번째 질의로 graph_score 가 아직 NULL 인 문서에 바닥값을 쓴다.
5. 바닥값보다 큰 문서를 세어 scored 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 2-3번을 안 풀면 평균에 넣을 점수가 없어 모든 문서가 바닥값이 된다. 그 경우를 먼저 알려 준다.
# 이 줄이 없으면 아래 scored 검사가 '평균 계산을 확인하세요' 라고 엉뚱한 원인을 지목한다
_pr = run_cypher('MATCH (n) WHERE n.pagerank IS NOT NULL RETURN count(n) AS c')[0]['c']
assert _pr > 0, 'pagerank 가 새겨진 개체가 없습니다. 2-3번을 먼저 푸세요'
# 2-2번을 건너뛰면 손으로 이어 준 한 편이 빠져 scored 가 하나 모자란다. 그것도 먼저 알려 준다
_mentions = run_cypher('MATCH (:Document)-[r:MENTIONS]->() RETURN count(r) AS c')[0]['c']
assert _mentions == 591, (f'MENTIONS 가 {_mentions}건입니다(591건이어야 합니다). 2-2번을 먼저 푸세요')
missing = run_cypher('MATCH (d:Document) WHERE d.graph_score IS NULL RETURN count(d) AS c')[0]['c']
assert missing == 0, f'graph_score 가 없는 논문이 {missing}편 있습니다. 바닥값을 채우세요'
assert scored == 59, (f'개체에서 점수를 받은 논문이 {scored}편입니다(59편이어야 합니다). 바닥값 0.15 **보다 큰** 문서를 세었는지(>= 가 아니라 >) 확인하세요')
sample = run_cypher('''MATCH (d:Document)-[:MENTIONS]->(e) WHERE e.pagerank IS NOT NULL
                       WITH d, avg(e.pagerank) AS mean
                       RETURN d.pmcid AS pmcid, mean, d.graph_score AS stored
                       ORDER BY d.pmcid LIMIT 5''')
for row in sample:
    assert abs(row['mean'] - row['stored']) < 1e-9, (f"{row['pmcid']} 의 graph_score 가 평균이 아닙니다. "
                                                    '최댓값이나 합계를 쓰지 않았는지 확인하세요')
print('✅ 통과!')

---
# 3. 그래프로 검색 결과 손보기

만들어 둔 그래프 점수와 커뮤니티로 검색 결과를 손보는 연습입니다(교안_02 2-3·3절). 하나는 **순서**를 바꾸고 하나는 **범위**를 자릅니다.

## 3-1. 두 점수를 섞어 다시 정렬하기
**배경**: 유사도와 그래프 점수는 **눈금이 달라** 그대로 섞으면 한쪽이 다 이깁니다(교안_02 2-3).

**요구사항**:
- **2-4번에서 `graph_score` 를 새로 새겼으므로, 1-1번의 검색을 그대로 다시 돌려 `hits` 를 새로 받으세요.** 1-1번 때 받아 둔 `hits` 에는 `graph_score` 가 아직 `None` 입니다.
- **`minmax(values)`**: 후보 안에서 최솟값을 0, 최댓값을 1 로 눌러 맞춘 **리스트**를 돌려줍니다. 모두 같으면 전부 `0.5` 입니다(교안_02 2-3).
- **`rerank(hits, weight)`**: **`(1 - weight) * 유사도 + weight * 그래프점수`** 로 섞어 다시 정렬한 `pmcid` 리스트를 돌려줍니다(교안_02 2-3).
- 새로 받은 `hits` 를 **`weight=0.3`** 으로 재정렬해 **`reranked`** 에 담고, 의미만의 순위와 나란히 출력하세요.
- `weight` 를 얼마나 올려야 순위가 바뀌는지 재세요. `0.00` 부터 `1.00` 까지 **`0.01` 씩** 올리며 `rerank` 를 돌려, 결과가 의미만의 순위(`top_ids`)와 **처음 달라지는 `weight`** 를 **`flip_weight`** 에 담으세요. 끝까지 안 달라지면 `None` 을 담으세요.

**확인 기준**: `reranked` 는 `top_ids` 와 **같은 다섯 편**이고 순서만 다릅니다. 자가채점은 여러분의 `rerank` 로 격자를 그 자리에서 다시 훑어 대조합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 최솟값과 최댓값으로 나눠 0에서 1 사이로 옮기고, 두 목록을 가중합해 정렬한다.
- flip_weight 는 격자를 훑는 반복문이다. 0.01 씩 올리며 결과가 달라지는 첫 자리에서 멈춘다.

세부구현:
1. minmax 는 최솟값·최댓값을 구해 각 값에서 최솟값을 빼고 범위로 나눈다. 범위가 0 이면 0.5 를 돌려준다.
2. rerank 는 hits 에서 두 점수 목록을 뽑아 각각 minmax 를 건다.
3. 두 목록을 zip 으로 짝지어 가중합을 만든다.
4. hits 와 가중합을 zip 해 가중합 내림차순으로 정렬하고 pmcid 만 돌려준다(sorted 는 안정 정렬이라 가중합이 같으면 유사도 순서가 유지된다).
5. flip_weight 는 0 부터 100 까지 정수를 돌며 100 으로 나눠 weight 를 만든다.
   0.01 을 거듭 더하면 부동소수 오차가 쌓여 격자가 어긋나므로, round(w, 2) 로 자리를 맞춘다.
6. 그 weight 로 rerank 를 돌려 top_ids 와 다르면 그 값을 담고 멈춘다. 끝까지 같으면 None 이다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 2-4번 뒤에 검색을 다시 돌렸는지부터 본다. 1-1번 때 받아 둔 낡은 hits 는 graph_score 가 None 이다
assert all(hit['graph_score'] is not None for hit in hits), \
    'hits 의 graph_score 가 비어 있습니다. 2-4번 뒤에 1-1번 검색을 다시 돌려 hits 를 새로 받으세요'
# 목록만 대조하면 이 셀의 리터럴을 옮겨 적어도 통과한다. 그 자리에서 직접 불러 본다
assert reranked == rerank(hits, 0.3), 'reranked 를 rerank(hits, 0.3) 의 결과 그대로 담으세요'
assert reranked[:3] == ['PMC13493261', 'PMC13495420', 'PMC13493376'], \
    (f'앞 세 편이 다릅니다: {reranked[:3]}. minmax 를 후보 다섯 편 안에서 했는지, '
     '(1-weight)*유사도 + weight*그래프점수 로 섞었는지 확인하세요')
assert set(reranked) == set(top_ids), '같은 다섯 편이어야 합니다. 후보를 바꾸지 마세요'
assert reranked != top_ids, '순서가 그대로입니다. 그래프 점수를 실제로 섞었는지 확인하세요'
assert minmax([2.0, 2.0, 2.0]) == [0.5, 0.5, 0.5], '모두 같은 값이면 0.5 를 돌려줘야 합니다'
assert all(abs(a - b) < 1e-9 for a, b in zip(minmax([1.0, 3.0, 2.0]), [0.0, 1.0, 0.5])), \
    'minmax 가 0과 1 사이로 맞추지 않습니다'
# 그래프 점수만 보게 하면(weight=1) 그 점수 순서가 그대로 나와야 한다
probe = [{'pmcid': 'A', 'score': 0.9, 'graph_score': 0.1},
         {'pmcid': 'B', 'score': 0.1, 'graph_score': 0.9}]
assert rerank(probe, 1.0) == ['B', 'A'], 'weight 를 실제로 쓰지 않았습니다'
assert rerank(probe, 0.0) == ['A', 'B'], 'weight=0 이면 유사도 순서 그대로여야 합니다'
# flip_weight 도 그 자리에서 같은 격자로 다시 훑어 대조한다.
# 격자를 만드는 방법(정수 나누기·누적 덧셈)에 따라 끝자리가 미세하게 달라지므로,
# == 가 아니라 허용 오차로 견준다
_swept = next((round(s / 100, 2) for s in range(101)
               if rerank(hits, round(s / 100, 2)) != top_ids), None)
_same = ((flip_weight is None and _swept is None)
         or (flip_weight is not None and _swept is not None
             and abs(flip_weight - _swept) < 1e-9))
assert _same, (f'flip_weight({flip_weight})가 직접 훑은 값({_swept})과 다릅니다. '
               '0.00 부터 0.01 씩 올리며 처음 달라지는 weight 를 담으세요')
assert flip_weight is not None and 0.20 <= flip_weight <= 0.30, \
    f'flip_weight 가 {flip_weight} 입니다. weight=0.00 부터 0.01 씩 차례로 확인했는지 보세요'
# 바로 앞 격자에서는 아직 안 뒤집혀야 '처음 달라지는' 자리다
assert rerank(hits, round(flip_weight - 0.01, 2)) == top_ids, \
    'flip_weight 바로 앞 격자에서 순위가 이미 달라졌습니다. 처음 달라지는 weight 를 담으세요'
print('✅ 통과!')

## 3-2. 커뮤니티로 검색 범위 좁히기
**배경**: 리랭킹은 순위를 **다듬을** 뿐 엉뚱한 논문을 빼지는 못합니다. **`Methotrexate`**(류마티스 관절염의 대표 약)와 같은 갈래 안에서만 찾아봅니다.

**요구사항**:
- 같은 투영(`drugGraph`)에 `gds.leiden.write` 로 개체마다 **`community`** 속성을 새기세요(34일차처럼 `randomSeed: 42` 와 `concurrency: 1` 을 함께 주세요).
- **`Methotrexate`** 노드의 `community` 값을 **`anchor_community`** 에 담으세요.
- 같은 질문으로 **상위 20편**을 받아, 그중 **그 묶음의 개체를 `MENTIONS` 로 가리키는 논문만** 유사도 내림차순으로 **`narrowed`**(pmcid 리스트)에 담으세요. 같은 논문이 여러 번 나오지 않게 하세요.

**확인 기준**: `narrowed` 는 중복이 없고 유사도 내림차순입니다. 1-1번의 1위 논문은 기준 약과 `MENTIONS` 로 이어져 있으므로 **그 약과 같은 묶음에 반드시 걸립니다.** 다만 **남는 편수는 정해져 있지 않아 자가채점은 편수를 세지 않습니다.** 대신 남긴 것이 모두 그 묶음에 걸리는지와 **뺀 것 중에 걸리는 게 없는지**를 함께 대조합니다.

> 씨앗을 줘도 준비 셀부터 다시 적재하면 노드 내부 번호가 바뀌어 묶음 경계가 갈립니다. 그래서 묶음 **번호**를 외우지 말고 그때그때 읽어 옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 커뮤니티를 노드 속성으로 쓴 뒤, 벡터 검색 결과에 MENTIONS 패턴을 이어 붙여 그 묶음에 든 것만 남긴다.

세부구현:
1. leiden.write 를 부른다. 인자 모양은 pageRank.write 와 같고 YIELD 로 쓸 컬럼만 받는다.
2. 기준 약의 Compound 노드를 이름으로 찾아 community 값을 꺼낸다.
3. SEARCH 절로 상위 20편을 잡은 다음, 같은 질의에 MATCH 한 줄을 더 붙여 MENTIONS 로 이어진 개체를 잡는다.
4. WHERE 로 그 개체의 community 가 기준값과 같은 것만 남기고, 중복을 없애 pmcid 를 순서대로 돌려받는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert anchor_community is not None, '커뮤니티를 찾지 못했습니다. leiden.write 를 먼저 부르세요'
# 학생이 담은 묶음 번호가 정말 그 약의 것인지부터 본다. 여기를 안 보면 다른 약으로 좁혀도 통과한다
assert anchor_community == run_cypher('MATCH (a {name:$n}) RETURN a.community AS c',
                                     n='Methotrexate')[0]['c'], \
    '기준 약의 묶음 번호가 아닙니다. 이름을 확인하세요'
assert len(narrowed) == len(set(narrowed)), '같은 논문이 여러 번 들어 있습니다. DISTINCT 를 쓰세요'
# 남는 편수는 세지 않는다(묶음이 크게 뭉친 실행에서는 20편이 다 남을 수도 있다). 아래 집합 대조가 대신한다
assert top_ids[0] in narrowed, \
    '기준 약과 MENTIONS 로 이어진 논문은 그 묶음에 걸리므로 반드시 남아야 합니다'

pool = [row['pmcid'] for row in run_cypher(
    '''MATCH (n:Document)
         SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 20) SCORE AS score
       RETURN n.pmcid AS pmcid ORDER BY score DESC''',
    q=embed_query('류마티스 관절염 치료에 쓰는 약'))]
assert [p for p in pool if p in set(narrowed)] == narrowed, \
    '상위 20편 안에서 유사도 내림차순 그대로 담으세요'

# 남긴 것은 전부 그 묶음에 걸려야 하고(건전성), 뺀 것은 하나도 안 걸려야 한다(완전성).
# 둘을 함께 봐야 필터가 맞았다고 할 수 있다. 편수를 맞히는 것이 목표가 아니라 세지 않는다
hit = {row['pmcid'] for row in run_cypher(
    '''MATCH (d:Document)-[:MENTIONS]->(e)
       WHERE d.pmcid IN $ids AND e.community = $c
       RETURN DISTINCT d.pmcid AS pmcid''', ids=pool, c=anchor_community)}
assert set(narrowed) == hit, (f'빠뜨린 논문 {sorted(hit - set(narrowed))} · '
                              f'잘못 남긴 논문 {sorted(set(narrowed) - hit)}')
print('✅ 통과!')

---
# 4. 찾은 근거로 답 만들기

리랭킹이 고른 논문을 **모델이 읽을 근거**로 바꾸고, 그 답을 어디까지 믿을지 가릅니다(교안_02 4절).

## 4-1. 발췌와 개체를 묶어 근거 만들기
**배경**: 근거는 논문 본문만으로 만들지 않고, 그 논문이 그래프에서 가리키는 개체를 함께 넣습니다.

**요구사항**:
- 교안_02 4-1 의 **`build_context(pmcids)`** 를 그대로 가져오세요. `pmcid` 리스트를 받아 **문자열 하나**를 돌려주고, 논문마다 아래 세 줄을 이 차례로 만들며, 논문과 논문 사이는 빈 줄 하나로 잇습니다.

```
[<pmcid>] <제목>
  본문: <본문 전체, 자르지 않음>
  이 논문이 언급한 개체: <이름 오름차순 최대 8개, 쉼표와 공백으로 이음>
```

- 논문은 **넘긴 순서 그대로** 이어 붙이고, 개체는 이름 오름차순으로 정렬한 뒤 앞 8개만 담으세요(같은 이름은 한 번만).
- 3-1번의 **`reranked` 앞 3편**으로 근거를 만들어 **`context`** 에 담고 앞 300자만 출력하세요.

**확인 기준**: `context` 는 문자열이고 대괄호로 감싼 `pmcid` 세 개가 순서대로 들어 있습니다. 둘째·셋째 줄 앞의 **공백 두 칸**도 문자열의 일부이고, 본문은 **원문 그대로** 들어갑니다(보기 좋게 하려고 개행을 공백으로 바꾸면 원문과 달라져 떨어집니다).

<details><summary>힌트</summary>

```text
접근방법:
- 세 논문을 한 번에 조회해 발췌와 개체 이름을 받고, 넘긴 순서대로 문단을 이어 붙인다.

세부구현:
1. WHERE d.pmcid IN $ids 로 세 편을 한 번에 잡고, OPTIONAL MATCH 로 MENTIONS 개체를 함께 받는다.
2. collect 앞에 ORDER BY 를 두어 이름 차례를 고정하고, 슬라이스로 앞 8개만 남긴다.
3. 조회 결과를 pmcid 로 찾을 수 있게 사전에 담고, 넘긴 리스트를 돌며 문단을 만든다.
4. 문단 사이는 빈 줄 하나로 잇는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(context, str) and context, 'context 는 비어 있지 않은 문자열이어야 합니다'
for _pmcid in reranked[:3]:
    assert f'[{_pmcid}]' in context, f'{_pmcid} 가 대괄호로 근거에 들어 있어야 합니다'
assert context.count('본문:') == 3, '논문마다 본문 줄을 한 줄씩 넣으세요'
assert context.count('이 논문이 언급한 개체:') == 3, '논문마다 개체 줄을 한 줄씩 넣으세요'
assert context.count('\n\n[PMC') == 2, '논문 사이는 빈 줄 하나로 이으세요'
# 본문은 자르기만 한 것이어야 한다. 개행을 공백으로 바꾸면 원문과 달라져 여기서 걸린다
_text = run_cypher('MATCH (d:Document {pmcid:$p}) RETURN d.text AS t', p=reranked[0])[0]['t']
assert _text in context, \
    '본문은 자르지 말고 통째로, 원문 그대로 넣으세요(개행을 공백으로 바꾸면 원문과 달라집니다)'
# in 검사만 하면 [..8] 을 빼고 23개를 다 담아도 앞 8개가 맞아 통과한다. 개체 수를 따로 센다
for _line in context.split('\n'):
    if _line.startswith('  이 논문이 언급한 개체:'):
        _names = _line.split(': ', 1)[1]
        assert len(_names.split(', ')) <= 8, \
            f'개체를 {len(_names.split(", "))}개 담았습니다. 이름 오름차순으로 앞 8개만 넣으세요'
print('✅ 통과!')

## 4-2. 근거를 붙여 답 받고 인용 대조하기
**배경**: 근거가 준비됐으니 모델에게 넘기고, 그 답이 근거 밖을 인용했는지 그 자리에서 대조합니다.

**요구사항**:
- 준비 셀이 준 **`answer_prompt`** 와 `ChatOpenAI(model='gpt-4o-mini', temperature=0)`, `StrOutputParser()` 를 **파이프로 이어** 체인을 만드세요(18일차 방식). 파서가 끝에 있으니 결과는 곧바로 문자열입니다.
- 질문 **`'류마티스 관절염 치료제로 무엇이 보고됐나요?'`** 과 4-1번의 `context` 를 빈칸 둘에 넣어 답을 받아 **`answer`**(문자열)에 담고 출력하세요.
- 답에서 `PMC` 로 시작하는 번호를 모두 찾아, 근거로 넘긴 세 편(`reranked[:3]`) 밖의 것만 골라 **정렬한 리스트**로 **`outside`** 에 담고 출력하세요(중복 없이, 근거 밖 인용이 없으면 빈 리스트).

**확인 기준**: `answer` 는 비어 있지 않은 문자열이고 대괄호로 감싼 `PMC` 번호가 들어 있습니다. `outside` 는 문자열 리스트이고 이번 근거에서는 대개 비어 있습니다. 답의 문장은 실행마다 달라지므로 자가채점은 **글자를 맞추지 않고 답의 모양과 여러분의 계산**만 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 틀과 모델과 파서를 파이프로 이어 체인을 만들고, 빈칸 둘을 딕셔너리로 넘긴다.

세부구현:
1. langchain_core.output_parsers 에서 파서를, langchain_openai 에서 모델을 가져온다.
2. 준비 셀의 틀과 모델, 파서를 파이프로 잇는다(18일차와 같다).
3. 체인의 invoke 에 context 와 question 두 칸을 딕셔너리로 넘긴다.
4. 답에서 PMC 로 시작하는 번호를 정규식으로 모두 찾아 집합으로 만들고, 근거 세 편을 뺀 뒤 정렬한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 모델의 문장은 실행마다 달라진다. 글자를 맞추지 않고 답의 모양과 여러분의 계산만 본다
import re as _re

assert isinstance(answer, str) and len(answer) > 40, \
    'answer 가 비어 있거나 너무 짧습니다. context 를 근거로 넘겼는지 확인하세요'
# 대괄호 인용은 모델의 몫이라 채점하지 않는다. 안 지켰으면 알려만 준다
if not _re.findall(r'\[PMC\d+\]', answer):
    print('⚠️ 답에 대괄호로 감싼 pmcid 가 없습니다. 모델이 규칙을 안 지킨 것이니 한 번 더 실행해 보세요')
# 여기가 채점 대상이다. 모델이 무엇을 인용했든 여러분의 집합 연산이 맞으면 통과한다
_want = sorted(set(_re.findall(r'PMC\d+', answer)) - set(reranked[:3]))
assert isinstance(outside, list), 'outside 는 리스트여야 합니다(집합이나 튜플이 아닙니다)'
assert outside == _want, (f'outside({outside})가 답에서 계산한 값({_want})과 다릅니다. '
                          '손으로 적지 말고 answer 에서 찾아 근거 세 편을 빼세요')
print('✅ 통과!')

## 4-3. 서술형: 그래프를 검색에 쓰는 두 방법, 그리고 그 답을 어디까지 믿을 것인가
**배경**: 앞의 네 갈래에서 만든 것을 놓고 네 가지를 판단합니다.

**요구사항**: 아래 네 가지를 각각 두세 문장으로 쓰세요.
1. 3-2번에서 빠진 논문을 하나 골라, 그 논문이 언급한 개체가 어느 묶음에 있는지 조회해 보고, 리랭킹이었다면 그 논문에 무슨 일이 일어났을지와 견줘 쓰세요.
2. 담당자가 "관련 논문을 **하나도 빠뜨리면 안 된다**"고 했습니다. 두 도구 중 무엇을 쓰지 말아야 하고 그 이유는 무엇인지.
3. 4-2번의 답에서 **문장 하나를 골라**, 그것이 논문이 보고한 것을 옮긴 것인지 효능이 입증됐다고 말하는 것인지 가르고 이유를 쓰세요(교안_02 4-3).
4. `outside` 가 비어 있지 않았다면 그 답을 어떻게 다뤄야 할까요. 그리고 **인용 대조가 잡지 못하는 것**은 무엇인지 쓰세요(교안_02 4-3).

아래 markdown 셀에 답을 쓰세요. 정답 노트북의 모범 서술과 비교해 보세요.

*(1~4번 각각에 대해 두세 문장씩 쓰세요)*